In [1]:
pip install pandas transformers accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# =========================================
# 🧠 TEXT-TO-SQL SYSTEM (LOCAL - Qwen 2.5B)
# =========================================

# Install once:
# pip install pandas transformers accelerate

# =========================================
# 📦 1. Imports
# =========================================
import sqlite3
import pandas as pd
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# =========================================
# 🤖 2. Load Open-Source LLM (Qwen)
# =========================================
MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # Use 2.5B if you have strong GPU

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=False
)

print("Model loaded successfully!")

# =========================================
# 🗄️ 3. CSV → SQLite Database
# =========================================
def csv_to_sqlite(csv_path, db_name="data.db"):
    conn = sqlite3.connect(db_name)

    df = pd.read_csv(csv_path)
    table_name = os.path.splitext(os.path.basename(csv_path))[0]

    df.to_sql(table_name, conn, if_exists="replace", index=False)

    return conn, table_name, df


# =========================================
# 🧠 4. Natural Language → SQL
# =========================================
def text_to_sql(question, table_name, columns):
    
    schema = f"Table: {table_name}\nColumns: {', '.join(columns)}"

    prompt = f"""
You are an expert SQL generator.

Convert the following natural language question into a correct SQLite query.

{schema}

Rules:
- Use only the given table and columns
- Do not invent column names
- Return ONLY SQL query

Question:
{question}

SQL:
"""

    output = generator(prompt)[0]["generated_text"]

    # Extract SQL safely
    sql = output.split("SQL:")[-1].strip()

    # Clean possible garbage
    sql = sql.split("\n")[0]

    return sql


# =========================================
# ⚡ 5. Execute SQL Query
# =========================================
def run_sql(conn, query):
    cursor = conn.cursor()

    try:
        cursor.execute(query)
        return cursor.fetchall()
    except Exception as e:
        return f"SQL Error: {e}"


# =========================================
# 🧾 6. Result → Human Explanation
# =========================================
def result_to_text(question, result):

    prompt = f"""
You are a data analyst.

User Question:
{question}

SQL Result:
{result}

Explain the result in simple and clear human language.
"""

    output = generator(prompt)[0]["generated_text"]

    explanation = output.split("language.")[-1].strip()

    return explanation


# =========================================
# 🚀 7. Full Pipeline
# =========================================
def text_to_sql_pipeline(csv_path, question):

    # Step 1: CSV → Database
    conn, table_name, df = csv_to_sqlite(csv_path)

    # Step 2: Extract schema
    columns = df.columns.tolist()

    # Step 3: Generate SQL
    sql_query = text_to_sql(question, table_name, columns)

    # Step 4: Execute SQL
    result = run_sql(conn, sql_query)

    # Step 5: Explain Result
    explanation = result_to_text(question, result)

    return {
        "question": question,
        "table": table_name,
        "sql_query": sql_query,
        "result": result,
        "explanation": explanation
    }


# =========================================
# 🧪 8. Run Example
# =========================================
if __name__ == "__main__":

    csv_path = "sample_employees.csv"   # <-- put your CSV here
    question = "What is the salary of each employee?"

    output = text_to_sql_pipeline(csv_path, question)

    print("\n==============================")
    print("🧠 QUESTION:")
    print(output["question"])

    print("\n📂 TABLE:")
    print(output["table"])

    print("\n📜 GENERATED SQL:")
    print(output["sql_query"])

    print("\n📊 RESULT:")
    print(output["result"])

    print("\n💡 EXPLANATION:")
    print(output["explanation"])
    print("==============================")

Loading model...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded successfully!


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧠 QUESTION:
What is the salary of each employee?

📂 TABLE:
sample_employees

📜 GENERATED SQL:
SELECT salary FROM sample_employees;

📊 RESULT:
[(50000,), (60000,), (55000,), (65000,), (52000,)]

💡 EXPLANATION:
The result shows the salary of each employee in a list format. The first number is the salary of the first employee, the second number is the salary of the second employee, and so on. The salary of each employee is listed in ascending order.
